## Spaceship Titanic, Transported Passenger Prediction

### Project Overview
This project addresses the Kaggle Spaceship Titanic competition. The task is to predict whether each passenger was transported to an alternate dimension based on passenger profile, travel details, and onboard activity data. It is a supervised binary classification problem.

### Problem Statement
Given labeled passenger records in the training set and unlabeled records in the test set, the objective is to learn the relationship between input features and transport outcome, then generate accurate predictions for unseen passengers.

### Objective
The objective is to build a high performing classification pipeline that maximizes predictive accuracy on the competition test set while maintaining strong internal validation discipline.

### Target Variable
The target variable is Transported. It is a binary outcome with two classes, True and False.

### Evaluation Metric
The competition uses accuracy as the scoring metric. Model development should therefore focus on improving classification correctness while validating performance carefully to avoid leaderboard overfitting.

### Analytical Focus
The main analytical focus is to determine which passenger attributes, travel variables, and behavioral signals contribute most to transport prediction. Particular attention should be given to missing data structure, categorical relationships, and hidden signal in composite fields.

### Project Goal
The goal is to produce a professional competition solution with strong feature understanding, reliable validation, and a final model that is competitive on the leaderboard.

In [1]:
# 1. IMPORT LIARIES

# core data handling
import pandas as pd
import numpy as np

# Display settings for easier data inspection
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [2]:
# 2. LOAD DATA

# Load the training and test datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sample_submission_df = pd.read_csv("sample_submission.csv")

In [3]:
# 3. SHAPE CHECK

# check the size of each dataset
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_submission_df.shape)

Train shape: (8693, 14)
Test shape: (4277, 13)
Sample submission shape: (4277, 2)


In [4]:
# 4. PREVIEW DATA

# preview the first few rows of each file
print("TRAIN DATA")
display(train_df.head())

print("TEST DATA")
display(test_df.head())

print("SAMPLE SUBMISSION")
display(sample_submission_df.head())

TRAIN DATA


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


TEST DATA


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name
0,0013_01,Earth,True,G/3/S,TRAPPIST-1e,27.0,False,0.0,0.0,0.0,0.0,0.0,Nelly Carsoning
1,0018_01,Earth,False,F/4/S,TRAPPIST-1e,19.0,False,0.0,9.0,0.0,2823.0,0.0,Lerome Peckers
2,0019_01,Europa,True,C/0/S,55 Cancri e,31.0,False,0.0,0.0,0.0,0.0,0.0,Sabih Unhearfus
3,0021_01,Europa,False,C/1/S,TRAPPIST-1e,38.0,False,0.0,6652.0,0.0,181.0,585.0,Meratz Caltilter
4,0023_01,Earth,False,F/5/S,TRAPPIST-1e,20.0,False,10.0,0.0,635.0,0.0,0.0,Brence Harperez


SAMPLE SUBMISSION


,PassengerId,Transported
0,0013_01,False
1,0018_01,False
2,0019_01,False
3,0021_01,False
4,0023_01,False


In [5]:
# 5. CHECK COLUMN NAMES

# view all column names
print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nSample submission columns:")
print(sample_submission_df.columns.tolist())

Train columns:
['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name', 'Transported']

Test columns:
['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']

Sample submission columns:
['PassengerId', 'Transported']


In [6]:
# 6. BASIC DATAFRAME INFO

# check data types and missing value presence at a high level
print("TRAIN INFO")
train_df.info()

print("\nTEST INFO")
test_df.info()

TRAIN INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB

TEST INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4277 entries, 0 to 4276
Data columns (total 13 columns):
 #   Column  